# SHIPP Stage 18 — Velocity Validation

**Priority:** P0  
**Purpose:** measure real end-to-end freshness from an eligible Lakebase Listing/Request change to downstream retrieval availability.

Spec SLO:

`retrieval_available_at - operational_change_at < 60 seconds`

This notebook **does not modify Lakebase**. For each measurement:
1. make one controlled eligible Listing or Request change in Lakebase;
2. let Lakebase CDF publish the event to Bronze;
3. run `90_run_pipeline`;
4. for a **Listing** measurement also run `91_run_rag_pipeline` so the AI Search index syncs;
5. run this notebook and record the result.

A miss is still valid evidence. Never discard runs over 60 seconds.

In [0]:
%run ../common/shipp_silver_lib

In [0]:
import json
import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F

from config.tables import (
    BRONZE_LISTINGS_HISTORY,
    BRONZE_REQUESTS_HISTORY,
    SILVER_LISTINGS,
    SILVER_REQUESTS,
    GOLD_CANDIDATE_MATCHES,
    GOLD_LISTING_SEARCH_DOCS,
    PIPELINE_RUN_LOG,
    VS_INDEX_NAME,
)

SLO_SECONDS = 60.0
VELOCITY_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.velocity_measurements"

dbutils.widgets.dropdown(
    "entity_type",
    "request",
    ["request", "listing"],
    "Entity type",
)
dbutils.widgets.text(
    "entity_id",
    "demo-request-001",
    "Entity ID",
)

ENTITY_TYPE = dbutils.widgets.get("entity_type").strip().lower()
ENTITY_ID = dbutils.widgets.get("entity_id").strip()

assert ENTITY_TYPE in {"request", "listing"}
assert ENTITY_ID, "BLOCKED: entity_id is required."

print("ENTITY_TYPE =", ENTITY_TYPE)
print("ENTITY_ID   =", ENTITY_ID)
print("SLO_SECONDS =", SLO_SECONDS)
print("TARGET      =", VELOCITY_TABLE)

## Gate 1 — Identify the actual operational change

The latest Bronze CDC event is used to identify the controlled source change.
`_timestamp` is the operational event timestamp carried by Lakebase CDC.

For the controlled demo, avoid unrelated writes to the same Bronze source while measuring.

In [0]:
if ENTITY_TYPE == "request":
    BRONZE_TABLE = BRONZE_REQUESTS_HISTORY
    SILVER_TABLE = SILVER_REQUESTS
    BUSINESS_KEY = "request_id"
else:
    BRONZE_TABLE = BRONZE_LISTINGS_HISTORY
    SILVER_TABLE = SILVER_LISTINGS
    BUSINESS_KEY = "listing_id"

bronze = spark.table(BRONZE_TABLE)

required_bronze = {
    BUSINESS_KEY,
    "_pg_change_type",
    "_pg_lsn",
    "_sort_by",
    "_timestamp",
}
missing_bronze = required_bronze - set(bronze.columns)
assert not missing_bronze, (
    f"BLOCKED: {BRONZE_TABLE} missing columns {sorted(missing_bronze)}"
)

latest_event = (
    bronze
    .filter(F.col(BUSINESS_KEY) == ENTITY_ID)
    .orderBy(
        F.col("_pg_lsn").desc(),
        F.col("_sort_by").desc(),
    )
    .first()
)

assert latest_event is not None, (
    f"BLOCKED: {ENTITY_ID} not found in {BRONZE_TABLE}."
)
assert latest_event["_pg_change_type"] != "delete", (
    "BLOCKED: latest event is DELETE. Use a current eligible entity "
    "for the Velocity measurement."
)

operational_change_at = latest_event["_timestamp"]

# Best available Bronze-arrival observation for this controlled test:
# first Delta commit at/after the event timestamp.
history_row = (
    spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}")
    .filter(F.col("timestamp") >= F.lit(operational_change_at))
    .orderBy(F.col("timestamp").asc())
    .select("timestamp", "version", "operation")
    .first()
)

bronze_at = history_row["timestamp"] if history_row else None

print("BRONZE_TABLE          =", BRONZE_TABLE)
print("Latest change type    =", latest_event["_pg_change_type"])
print("Latest _pg_lsn        =", latest_event["_pg_lsn"])
print("Latest _sort_by       =", latest_event["_sort_by"])
print("operational_change_at =", operational_change_at)
print("bronze_at             =", bronze_at)
if history_row:
    print("bronze Delta version  =", history_row["version"])
    print("bronze operation      =", history_row["operation"])

print("PASS Gate 1 — controlled operational change identified.")

## Gate 2 — Prove the same entity reached Silver and Gold

This gate rejects stale outputs. The Silver and Gold processing timestamps must be at or after the selected operational change.

In [0]:
silver = spark.table(SILVER_TABLE)
assert "silver_processed_at" in silver.columns, (
    f"BLOCKED: {SILVER_TABLE} has no silver_processed_at."
)

silver_row = (
    silver
    .filter(F.col(BUSINESS_KEY) == ENTITY_ID)
    .orderBy(F.col("silver_processed_at").desc())
    .select(BUSINESS_KEY, "silver_processed_at")
    .first()
)
assert silver_row is not None, (
    f"BLOCKED: {ENTITY_ID} has not reached {SILVER_TABLE}."
)

silver_at = silver_row["silver_processed_at"]
assert silver_at >= operational_change_at, (
    "BLOCKED: Silver is older than the selected operational change. "
    "Run notebooks/pipeline/90_run_pipeline first."
)

gold = spark.table(GOLD_CANDIDATE_MATCHES)
gold_scope = (
    gold.filter(F.col("request_id") == ENTITY_ID)
    if ENTITY_TYPE == "request"
    else gold.filter(F.col("listing_id") == ENTITY_ID)
)

gold_row = (
    gold_scope
    .agg(F.max("gold_processed_at").alias("gold_at"))
    .first()
)
gold_at = gold_row["gold_at"] if gold_row else None

assert gold_at is not None, (
    "BLOCKED: no Gold candidate row exists for this entity. "
    "Use an eligible demo entity that produces at least one match."
)
assert gold_at >= operational_change_at, (
    "BLOCKED: Gold is older than the selected operational change. "
    "Run notebooks/pipeline/90_run_pipeline."
)

pipeline_log = spark.table(PIPELINE_RUN_LOG)
gold_run = (
    pipeline_log
    .filter(
        (F.col("stage") == "40_gold_matches")
        & (F.upper(F.col("status")) == "PASS")
        & (F.col("ended_at") >= F.lit(operational_change_at))
    )
    .orderBy(F.col("ended_at").asc())
    .first()
)

pipeline_run_id = gold_run["run_id"] if gold_run else None

print("silver_at       =", silver_at)
print("gold_at         =", gold_at)
print("pipeline_run_id =", pipeline_run_id)
print("PASS Gate 2 — entity reached current Silver and Gold.")

## Gate 3 — Determine retrieval availability

- **Request measurement:** Gold Candidate Matches are the downstream retrieval product, so `gold_at` is the retrieval timestamp.
- **Listing measurement:** the measurement additionally requires the triggered AI Search sync. Notebook 54 exits only after the index is synced and a semantic probe succeeds. We use that successful Stage 54 end time, then perform a live filtered query for this exact Listing ID.

In [0]:
search_at = None
rag_run_id = None
retrieval_mode = "gold_candidate_matches"

if ENTITY_TYPE == "request":
    retrieval_available_at = gold_at
else:
    from databricks.sdk import WorkspaceClient
    from rag.retrieval import rows_from_query_response

    docs = spark.table(GOLD_LISTING_SEARCH_DOCS)
    doc = (
        docs
        .filter(F.col("listing_id") == ENTITY_ID)
        .select("listing_id", "title", "search_text")
        .first()
    )
    assert doc is not None, (
        f"BLOCKED: {ENTITY_ID} is not in {GOLD_LISTING_SEARCH_DOCS}. "
        "Run the RAG pipeline after the controlled Listing change."
    )

    rag_run = (
        pipeline_log
        .filter(
            (F.col("stage") == "54_ai_search_index")
            & (F.upper(F.col("status")) == "PASS")
            & (F.col("ended_at") >= F.lit(operational_change_at))
        )
        .orderBy(F.col("ended_at").asc())
        .first()
    )
    assert rag_run is not None, (
        "BLOCKED: no successful Stage 54 run exists after this Listing change. "
        "Run notebooks/pipeline/91_run_rag_pipeline, then rerun this notebook."
    )

    search_at = rag_run["ended_at"]
    rag_run_id = rag_run["run_id"]

    query_text = (
        (doc["title"] or "").strip()
        or (doc["search_text"] or "")[:300].strip()
    )
    assert query_text, "BLOCKED: search document has no usable query text."

    w = WorkspaceClient()
    response = w.vector_search_indexes.query_index(
        index_name=VS_INDEX_NAME,
        columns=["listing_id", "title"],
        query_text=query_text,
        filters_json=json.dumps({"listing_id": ENTITY_ID}),
        num_results=1,
    )
    rows = rows_from_query_response(response.as_dict())

    assert rows and rows[0].get("listing_id") == ENTITY_ID, (
        f"BLOCKED: {ENTITY_ID} is not retrievable from AI Search yet."
    )

    retrieval_available_at = search_at
    retrieval_mode = "ai_search"

print("retrieval_mode         =", retrieval_mode)
print("search_at              =", search_at)
print("rag_run_id             =", rag_run_id)
print("retrieval_available_at =", retrieval_available_at)
print("PASS Gate 3 — downstream retrieval availability proven.")

## Gate 4 — Calculate and persist the measurement

A run over 60 seconds is **not discarded**. It is stored with `slo_met = false`, exactly as required by the SHIPP failure policy.

In [0]:
latency_seconds = (
    retrieval_available_at - operational_change_at
).total_seconds()

assert latency_seconds >= 0, (
    "FAILED: negative Velocity latency indicates inconsistent timestamps."
)

slo_met = latency_seconds < SLO_SECONDS
run_id = str(uuid.uuid4())
measured_at = datetime.now(timezone.utc).replace(tzinfo=None)

MEASUREMENT_SCHEMA = """
run_id string,
entity_type string,
entity_id string,
retrieval_mode string,
operational_change_at timestamp,
bronze_at timestamp,
silver_at timestamp,
gold_at timestamp,
search_at timestamp,
retrieval_available_at timestamp,
latency_seconds double,
slo_met boolean,
pipeline_run_id string,
rag_run_id string,
measured_at timestamp
"""

measurement = [(
    run_id,
    ENTITY_TYPE,
    ENTITY_ID,
    retrieval_mode,
    operational_change_at,
    bronze_at,
    silver_at,
    gold_at,
    search_at,
    retrieval_available_at,
    float(latency_seconds),
    bool(slo_met),
    pipeline_run_id,
    rag_run_id,
    measured_at,
)]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

(
    spark.createDataFrame(measurement, MEASUREMENT_SCHEMA)
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(VELOCITY_TABLE)
)

print("run_id                  =", run_id)
print("latency_seconds          =", round(latency_seconds, 3))
print("SLO < 60 seconds met?    =", slo_met)
print(
    "PASS Gate 4 — measurement recorded."
    if slo_met
    else "PASS Gate 4 — measurement recorded as an SLO MISS."
)

display(
    spark.table(VELOCITY_TABLE)
    .filter(F.col("run_id") == run_id)
)

## Gate 5 — Evidence summary

For the final submission, use multiple controlled runs. The rubric requires measured evidence; do not claim the SLO from a single fast component or from an unmeasured assumption.

In [0]:
measurements = spark.table(VELOCITY_TABLE)

summary = (
    measurements
    .agg(
        F.count("*").alias("runs"),
        F.expr("percentile_approx(latency_seconds, 0.5)").alias("median_seconds"),
        F.max("latency_seconds").alias("worst_seconds"),
        F.sum(F.when(F.col("slo_met"), 1).otherwise(0)).alias("slo_met_runs"),
    )
    .first()
)

print("=== SHIPP VELOCITY EVIDENCE ===")
print("Measured runs :", summary["runs"])
print("Median sec    :", summary["median_seconds"])
print("Worst sec     :", summary["worst_seconds"])
print("SLO-met runs  :", summary["slo_met_runs"])

if summary["runs"] < 5:
    print(
        "NOTE — final evidence target is 5 controlled measurements; "
        f"{5 - summary['runs']} more run(s) remain."
    )

display(
    measurements.orderBy(F.col("measured_at").desc()).limit(20)
)

print(json.dumps({
    "table": VELOCITY_TABLE,
    "runs": summary["runs"],
    "median_seconds": summary["median_seconds"],
    "worst_seconds": summary["worst_seconds"],
    "slo_met_runs": summary["slo_met_runs"],
    "status": "PASS",
}, indent=2, default=str))